# 03 · Will a Bike or Free Dock Be Available at Arrival?

**probability-based BIXI station availability experiment.**

Imagine cycling toward a station that currently has one free dock. Is there a reasonable
chance that at least one dock will remain **15 minutes later**? The project combines
short-horizon demand forecasts with current bike/dock inventory to estimate that chance.



## 1. Prepare inputs

After creating CatBoost models from `02_Demand_Forecasting.ipynb`. This experiment also
requires time-stamped GBFS observations in `data/processed/bixi_live_snapshots.csv`.



In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import time

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from catboost import CatBoostRegressor
from scipy.stats import skellam
from sklearn.metrics import accuracy_score, brier_score_loss
from IPython.display import display

root = next((p for p in [Path.cwd(), Path.cwd() / "notebook",
                         Path.cwd().parent, Path.cwd().parent / "notebook"]
             if (p / "data" / "processed").is_dir()), None)
if root is None:
    raise FileNotFoundError("Run from the BIXI project directory containing data/processed.")

processed = root / "data" / "processed"
snapshots_path = processed / "bixi_live_snapshots.csv"
station_info_path = processed / "bixi_station_information_at_collection.csv"
features = ["station", "year", "month", "day_of_week", "hour", "time_of_day_slot"]
status_url = "https://gbfs.velobixi.com/gbfs/en/station_status.json"
info_url = "https://gbfs.velobixi.com/gbfs/en/station_information.json"

model_paths = {
    "arrivals": root / "models" / "bixi_arrivals_2022_2024.cbm",
    "departures": root / "models" / "bixi_departures_2022_2024.cbm",
}
missing = [str(p) for p in model_paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Train/load the CatBoost models before this notebook: {missing}")

models = {}
for name, path in model_paths.items():
    model = CatBoostRegressor()
    model.load_model(str(path))
    models[name] = model
print("Loaded model feature order:", models["arrivals"].feature_names_)

## 2. Optional data collection (disabled by default)

To build a new validation dataset, BIXI's GBFS station-status snapshots are collected every
five minutes. Save station information **at the same time** because station names and locations
may change later. A short eight-snapshot window lasts approximately 35 minutes and is not
enough for production-level evaluation.

When `COLLECT_NEW_SNAPSHOTS = False`, the cell never calls the public API.
If you collect again, use a new output filename or archive the previous CSV first.

In [ ]:
COLLECT_NEW_SNAPSHOTS = False
SNAPSHOT_COUNT = 8
INTERVAL_SECONDS = 300

if COLLECT_NEW_SNAPSHOTS:
    if snapshots_path.exists():
        raise FileExistsError(
            f"{snapshots_path} exists. Rename/archive it before collecting a fresh window."
        )
    info_response = requests.get(info_url, timeout=20)
    info_response.raise_for_status()
    pd.DataFrame(info_response.json()["data"]["stations"]).to_csv(
        station_info_path, index=False
    )

    for attempt in range(SNAPSHOT_COUNT):
        response = requests.get(status_url, timeout=20)
        response.raise_for_status()
        current = pd.DataFrame(response.json()["data"]["stations"])
        required = ["station_id", "num_bikes_available", "num_docks_available",
                    "is_renting", "is_returning", "last_reported"]
        missing_fields = set(required).difference(current.columns)
        if missing_fields:
            raise KeyError(f"Missing GBFS station-status fields: {missing_fields}")
        current = current[required].copy()
        current["collected_at"] = datetime.now(timezone.utc).isoformat()
        current.to_csv(snapshots_path, mode="a", index=False,
                       header=not snapshots_path.exists())
        print(f"Snapshot {attempt + 1}/{SNAPSHOT_COUNT}: {len(current)} stations")
        if attempt < SNAPSHOT_COUNT - 1:
            time.sleep(INTERVAL_SECONDS)

## 3. Load snapshots and audit data freshness



In [ ]:
if not snapshots_path.is_file():
    raise FileNotFoundError(
        f"{snapshots_path} is missing. Restore the original collection or set "
        "COLLECT_NEW_SNAPSHOTS=True above."
    )

raw = pd.read_csv(snapshots_path, dtype={"station_id": "string"})
required = {"station_id", "num_bikes_available", "num_docks_available",
            "is_renting", "is_returning", "last_reported", "collected_at"}
assert required.issubset(raw.columns), required - set(raw.columns)

raw["collected_at"] = pd.to_datetime(raw["collected_at"], utc=True)
raw["last_reported"] = pd.to_datetime(raw["last_reported"], unit="s", utc=True,
                                      errors="coerce")
raw["status_age_min"] = (raw["collected_at"] - raw["last_reported"]).dt.total_seconds() / 60
raw = raw.sort_values(["station_id", "collected_at"]).drop_duplicates(
    ["station_id", "collected_at"], keep="last"
).reset_index(drop=True)

print("Raw snapshots:", len(raw))
print("Stations:", raw.station_id.nunique())
print("Collection times:", raw.collected_at.nunique())
print("Age 0–30 min:", raw.status_age_min.between(0, 30).sum())
display(raw.status_age_min.describe(percentiles=[.5, .9, .95]))

## 4. Construct future outcomes 
Use future snapshots **only** to create labels after predictions are formed from current
features. A 15-minute target corresponds to three observed five-minute steps, or six for
30 minutes. 

In [ ]:
def make_targets(frame, horizon_min, nominal_step_min=5, tolerance_min=2):
    if horizon_min % nominal_step_min:
        raise ValueError("Horizon must be a multiple of nominal polling interval.")
    steps = horizon_min // nominal_step_min
    data = frame.copy()
    grouped = data.groupby("station_id", sort=False)
    for column in ["collected_at", "num_bikes_available", "num_docks_available",
                   "is_renting", "is_returning", "status_age_min"]:
        data[f"future_{column}"] = grouped[column].shift(-steps)
    elapsed = (data.future_collected_at - data.collected_at).dt.total_seconds() / 60
    start_fresh = data.status_age_min.between(0, 30)
    end_fresh = data.future_status_age_min.between(0, 30)
    operational = (
        (data.is_renting == 1) & (data.is_returning == 1)
        & (data.future_is_renting == 1) & (data.future_is_returning == 1)
    )
    data = data[(elapsed - horizon_min).abs().le(tolerance_min)
                & start_fresh & end_fresh & operational].copy()
    data["horizon_min"] = horizon_min
    data["actual_bike_available"] = (data.future_num_bikes_available > 0).astype(int)
    data["actual_dock_available"] = (data.future_num_docks_available > 0).astype(int)
    data["baseline_bike_available"] = (data.num_bikes_available > 0).astype(int)
    data["baseline_dock_available"] = (data.num_docks_available > 0).astype(int)
    return data


validation = pd.concat([make_targets(raw, horizon) for horizon in (15, 30)],
                       ignore_index=True)
if validation.empty:
    raise ValueError("No valid future observations; collect a longer or fresher window.")
display(validation.groupby("horizon_min").agg(
    examples=("station_id", "size"), stations=("station_id", "nunique"),
    bike_available_fraction=("actual_bike_available", "mean"),
    dock_available_fraction=("actual_dock_available", "mean")
))

## 5. Resolve station identity for the historical model

Station-status snapshots identify stations by ID, but historical trip models use matched
station **names**. Use station-information metadata captured during the same observation
period. This stage accepts only exact normalized name matches 

In [ ]:
if not station_info_path.is_file():
    raise FileNotFoundError(
        f"Station metadata missing: {station_info_path}. Save the station-information "
        "feed from the observation period; avoid substituting a later snapshot without auditing."
    )

station_info = pd.read_csv(station_info_path, dtype={"station_id": "string"})
station_info["station"] = station_info["name"].astype(str).str.lower().str.strip()
assert not station_info.station_id.duplicated().any()

con = duckdb.connect()
train_paths = [str(processed / f"bixi_{y}_complete.parquet") for y in (2022, 2023, 2024)]
missing_train = [p for p in train_paths if not Path(p).is_file()]
if missing_train:
    raise FileNotFoundError(f"Missing historical station-name parquet files: {missing_train}")
known_names = set(con.execute("""
    SELECT DISTINCT station FROM read_parquet(?)
""", [train_paths]).df().station.astype(str).str.lower().str.strip())

station_info["matched_station"] = station_info.station.where(station_info.station.isin(known_names))
if station_info.matched_station.dropna().duplicated().any():
    raise ValueError("Multiple current IDs map to one historical station name; review manually.")

mapped = validation.merge(
    station_info[["station_id", "matched_station"]],
    on="station_id", how="left", validate="many_to_one"
)
print("Valid outcomes:", len(mapped))
print("With historical match:", mapped.matched_station.notna().sum())
display(mapped.groupby(["horizon_min", mapped.matched_station.notna()]).size())

## 6. Convert expected flows into availability probabilities

For each *current* 15-minute station observation, the two trained CatBoost models estimate
mean arrivals (λₐ) and departures (λᵈ). As a first approximation, assume independent
Poisson flows, so net bike change `arrivals − departures` follows a Skellam distribution.

Given currently available bikes `B` and docks `D`:

- `P(bike at arrival) = P(B + net_change ≥ 1)`
- `P(dock at arrival) = P(D − net_change ≥ 1)`



In [ ]:
eval_15 = mapped[(mapped.horizon_min == 15) & mapped.matched_station.notna()].copy()
if eval_15.empty:
    raise ValueError("No 15-minute examples have a historical station match.")

local = eval_15.collected_at.dt.tz_convert("America/Toronto")
X = pd.DataFrame({
    "station": eval_15.matched_station.astype(str).to_numpy(),
    "year": local.dt.year.to_numpy(),
    "month": local.dt.month.to_numpy(),
    "day_of_week": ((local.dt.dayofweek + 1) % 7).to_numpy(),
    "hour": local.dt.hour.to_numpy(),
    "time_of_day_slot": (local.dt.hour * 4 + local.dt.minute // 15).to_numpy(),
})[features]

lambda_a = np.maximum(models["arrivals"].predict(X), 1e-8)
lambda_d = np.maximum(models["departures"].predict(X), 1e-8)
bikes = eval_15.num_bikes_available.to_numpy(dtype=int)
docks = eval_15.num_docks_available.to_numpy(dtype=int)

# Net flow N = arrivals - departures. Events are N >= 1 - B or N <= D - 1.
eval_15["prob_bike_available"] = 1 - skellam.cdf(-bikes, lambda_a, lambda_d)
eval_15["prob_dock_available"] = skellam.cdf(docks - 1, lambda_a, lambda_d)
for resource in ("bike", "dock"):
    eval_15[f"prob_{resource}_available"] = eval_15[
        f"prob_{resource}_available"
    ].clip(0, 1)
    eval_15[f"pred_{resource}_available"] = (
        eval_15[f"prob_{resource}_available"] >= .5
    ).astype(int)

display(eval_15[["station_id", "num_bikes_available", "num_docks_available",
                 "prob_bike_available", "prob_dock_available"]].head())

## 7. Evaluation: probability quality and operational edge cases

We compare the model with a **persistence** prediction that assumes today's bike/dock
availability state remains unchanged for 15 minutes. Its probability is 0 or 1.

The **Brier score** measures mean squared error between a probability and the actual
binary outcome. Lower is better.

In [ ]:
rows = []
for resource, count_col in [("bike", "num_bikes_available"),
                            ("dock", "num_docks_available")]:
    for cohort, mask in [
        ("All matched stations", np.ones(len(eval_15), dtype=bool)),
        ("0–2 remaining", eval_15[count_col].le(2)),
        ("1–2 remaining", eval_15[count_col].between(1, 2)),
        ("0 remaining", eval_15[count_col].eq(0)),
    ]:
        subset = eval_15.loc[mask]
        if subset.empty:
            continue
        actual = subset[f"actual_{resource}_available"]
        persist = subset[f"baseline_{resource}_available"]
        probability = subset[f"prob_{resource}_available"]
        predicted = subset[f"pred_{resource}_available"]
        rows.append({
            "resource": resource, "cohort": cohort, "n": len(subset),
            "actual_availability_pct": 100 * actual.mean(),
            "persistence_accuracy_pct": 100 * accuracy_score(actual, persist),
            "prototype_accuracy_pct": 100 * accuracy_score(actual, predicted),
            "persistence_brier": brier_score_loss(actual, persist),
            "prototype_brier": brier_score_loss(actual, probability),
        })
evaluation = pd.DataFrame(rows)
display(evaluation.round(4))

## 8. Does risk ranking help at nearly full stations?



In [ ]:
near_full = eval_15[eval_15.num_docks_available.between(1, 2)].copy()
if len(near_full) < 20:
    print("Not enough near-full examples for useful risk quartiles:", len(near_full))
else:
    near_full["predicted_failure_risk"] = 1 - near_full.prob_dock_available
    near_full["actual_failure"] = 1 - near_full.actual_dock_available
    near_full["risk_group"] = pd.qcut(
        near_full.predicted_failure_risk.rank(method="first"), q=4,
        labels=["Lowest", "Lower middle", "Upper middle", "Highest"]
    )
    risk = near_full.groupby("risk_group", observed=True).agg(
        n=("actual_failure", "size"),
        predicted_failure_pct=("predicted_failure_risk", lambda x: 100 * x.mean()),
        observed_failure_pct=("actual_failure", lambda x: 100 * x.mean()),
    )
    display(risk.round(2))
    ax = risk[["predicted_failure_pct", "observed_failure_pct"]].plot(
        kind="bar", figsize=(9, 4), rot=0, color=["#7E9EBA", "#C85C56"]
    )
    ax.set_ylabel("No dock on arrival (%)")
    ax.set_xlabel("Predicted dock-failure risk quartile")
    ax.set_title("Near-full stations: exploratory risk ranking")
    ax.grid(axis="y", alpha=.2)
    plt.tight_layout()
    plt.show()